# 07 · Landmark and incident-SMM analyses — Severe maternal morbidity MIMIC-IV (M2)

| Field | Content |
|---|---|
| Notebook sequence | 01: S1 · S2 · S3 · S4 · G1 → 02: S5 · G2 → 03: S6 · S7 · G3 · R1 · S8 → 04: S8 → 05: S9 (S7) → 06: S9 (S3) → **07: S9 · C1-C3 · R2 · R4** → 08: S9 · C4 → 09: S9 · C4 → 10: G4 · S10 · S11 |
| What it does | Re-runs the main pipeline on populations and labels that remove morbidity already present at T = admission + 24 h (criteria C1 and C2), tests detection bias (C3), an alternative SMM definition, richer predictors (return R4), escalation of care after T (return R2), race and ethnicity by group (S10), gate G3 for the R2 outcome and the sample-size reduction |
| Stages and gates | S9 · C1 · C2 · C3 · R4 → S7 · R2 → S3 · G3 · S10 |
| Inputs | Local PostgreSQL MIMIC-IV database (tables of `sql/`): tables of notebook 03 plus `timing_flags_v3`, `smm_code_hits_v3`, `escalation_v3` (`sql/07_timing_audit/`) |
| Outputs | `data/outputs_<analysis>.txt` and the summaries `outputs_richer_predictors_summary.txt`, `outputs_escalation_summary.txt`, `outputs_c3_labs_summary.txt`, `outputs_g3_escalation.txt`, `outputs_escalation_reduced*.txt` |
| Data use | Only aggregate results are shown and saved; nothing at patient level. Do not add cells that show or export rows |
| Time | Several hours in total; each section can be run alone with the `RUN_...` switches |
| Next | `08_era_temporal_validation_smm.ipynb` |
| Spanish version | `07_landmark_incident_smm_es.ipynb` (same code) |

| Timing class | Rule (SMM admissions only) |
|---|---|
| `1 prevalent` | a procedure indicator dated before T, **or** ICU admission before T, **or** renal-failure code with laboratory AKI onset at or before T |
| `2 indeterminate` | none of the above, and a procedure indicator on the calendar day that contains T |
| `3 incident (documented)` | none of the above, and a procedure indicator after T **or** renal-failure code with laboratory AKI onset after T |
| `4 untimed` | the rest (diagnosis codes only, no laboratory timing) |

Admissions are removed, never relabelled as non-events. Run `sql/07_timing_audit/01_timing_flags_v3.sql` first.

> **Run order.** Run all cells top to bottom. First `QUICK = True` for a smoke test (minutes; never report QUICK numbers), then `QUICK = False`.

## S1 · Governance and configuration — libraries, connection and settings

Fixes the seed, the outer and inner folds, the repeats, the bootstrap replicates and the reference counts `EXPECTED` frozen at gate G1. `QUICK = True` is a smoke test (minutes); its numbers are never reported.

In [ ]:
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# notebooks/ -> project root is one level up
sys.path.append(os.path.abspath(os.path.join("..")))
import libraries.database as db_lib
engine = db_lib.get_connect_database()

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

# ----------------------------------------------------------------------------- settings
QUICK        = False         # True = smoke test (small forests, 1 repeat, B = 50). Never report QUICK numbers.
SEED         = 20260101
OUTER_FOLDS  = 5
N_REPEATS    = 3            # repeats of the outer grouped CV (development instability)
INNER_FOLDS  = 3
BOOT         = 1000         # cluster bootstrap replicates for the main analysis
BOOT_SENS    = 300          # replicates for the sensitivity analyses
N_JOBS       = -1
N_TREES      = 300
XGB_TREES    = 200
APPLY_LIMITS = True         # set values outside the plausibility limits of the dictionary to missing
CAT_COLS     = ["ethnicity", "insurance", "language", "admission_type", "admission_location"]
DCA_T        = [0.005, 0.01, 0.02, 0.03, 0.05, 0.10, 0.15, 0.20]
EXPECTED     = {"eligible_deliveries": 19383, "smm": 445, "deaths": 3}   # from the audit counts (Q-F / Q-G)

if QUICK:
    OUTER_FOLDS, N_REPEATS, BOOT, BOOT_SENS = 3, 1, 50, 30
    N_TREES, XGB_TREES = 60, 50
print(f"QUICK={QUICK}  outer={OUTER_FOLDS}x{N_REPEATS}  inner={INNER_FOLDS}  B={BOOT}  trees={N_TREES}")

# ----------------------------------------------------------------------------- language and presentation
# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
# Nothing below changes a computation: it only decides how tables and messages are shown. The English aggregate
# reports data/outputs_*.txt are written exactly as before (the article cites them); the Spanish notebook also
# writes a translated copy, data/outputs_*_es.txt.
LANG = "en"
import builtins, contextlib, importlib, io, html as _html
import numpy as np
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)
def _tr(s, lang=None):
    """Translate one line keeping its indentation and trailing spaces."""
    lang = lang or LANG
    if lang != "es" or not isinstance(s, str):
        return s
    out = []
    for ln in s.split("\n"):
        core = ln.strip()
        if not core:
            out.append(ln); continue
        i = ln.index(core)
        out.append(ln[:i] + translate(core, "es") + ln[i + len(core):])
    return "\n".join(out)

_REC = None          # while emit() runs: the blocks of the English report, [("text", line) | ("table", frame, index)]

def print(*args, sep=" ", end="\n", file=None, flush=False):
    """Inside emit() the English text goes to the report unchanged; outside it, messages follow LANG."""
    if _REC is not None and file is None:
        _REC.append(("text", sep.join(str(a) for a in args)))
        return builtins.print(*args, sep=sep, end=end, flush=flush)
    if LANG == "es":
        args = [_tr(a, "es") for a in args]
    return builtins.print(*args, sep=sep, end=end, file=file, flush=flush)

def _cell_fmt(v):
    # Display format of a cell: thousands separator for counts, 4 significant digits for small values.
    if isinstance(v, (bool, np.bool_)) or v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        if float(v).is_integer() and abs(v) >= 1000:
            return f"{int(v):,}"
        return f"{v:.4g}" if abs(v) < 1 else f"{v:,.4f}".rstrip("0").rstrip(".")
    return v

def _html_table(df, index):
    df = df.astype(object)
    df = df.map(_cell_fmt) if hasattr(df, "map") else df.applymap(_cell_fmt)
    return f"<div style='max-height:420px; overflow:auto'>{df.to_html(index=index, na_rep='', border=0)}</div>"

def _in_notebook():
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except ImportError:
        return False

def table(df, index=False, title=None):
    """Show a table. Inside emit() it also prints the English text of the report (df.to_string)."""
    if isinstance(df, pd.Series):
        df = df.to_frame()
        index = True
    if _REC is not None:
        _REC.append(("table", df, index))
        return builtins.print(df.to_string(index=index))
    if not _in_notebook():
        return builtins.print(translate(df, LANG).to_string(index=index))
    from IPython.display import HTML, display as _display
    head = f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>" if title else ""
    _display(HTML(head + _html_table(translate(df, LANG), index)))

def _blocks_html(blocks):
    """Report blocks as HTML: a text line right before a table is its title; rules of '=' are dropped."""
    out, pending = [], []
    def flush_text():
        for ln in pending:
            if ln.strip():
                out.append(f"<div style='white-space:pre-wrap; font-family:monospace; margin:2px 0'>{_html.escape(_tr(ln.rstrip()))}</div>")
        pending.clear()
    for b in blocks:
        if b[0] == "text":
            pending.extend(l for l in b[1].split("\n") if not set(l.strip()) <= {"="} or not l.strip())
        else:
            title = None
            while pending and not pending[-1].strip():
                pending.pop()
            if pending:
                title = pending.pop().strip()
            flush_text()
            if title:
                out.append(f"<p style='margin:14px 0 4px 0'><b>{_html.escape(_tr(title))}</b></p>")
            out.append(_html_table(translate(b[1], LANG), b[2]))
    flush_text()
    return "\n".join(out)

def _blocks_text_es(blocks):
    """Spanish copy of a report (data/outputs_*_es.txt)."""
    lines = []
    for b in blocks:
        if b[0] == "text":
            lines.extend(_tr(l, "es") for l in b[1].split("\n"))
        else:
            lines.extend(translate(b[1], "es").to_string(index=b[2]).split("\n"))
    return "\n".join(lines) + "\n"

def _emit_to(path, fn, show=True):
    """Run fn, write its English text to `path` exactly as printed and, from the Spanish notebook, a translated copy
    (*_es.txt); show the report as tables."""
    global _REC
    buf = io.StringIO()
    _REC = []
    try:
        with contextlib.redirect_stdout(buf):
            fn()
    finally:
        blocks, _REC = _REC, None
    text = buf.getvalue()
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(text)
    if show:
        show_report(blocks)
    if LANG == "es":
        with open(path.replace(".txt", "_es.txt"), "w", encoding="utf-8") as fh:
            fh.write(_blocks_text_es(blocks))
    return text

def show_report(blocks):
    if _in_notebook():
        from IPython.display import HTML, display as _display
        _display(HTML(_blocks_html(blocks)))
    else:
        for b in blocks:
            builtins.print(_tr(b[1]) if b[0] == "text" else translate(b[1], LANG).to_string(index=b[2]))

def notebook_summary(rows, title="Notebook summary"):
    """Summary of the notebook (as in the neonatal repository): one value per row; a value that cannot be computed
    (for example, an analysis that was not run in this session) is shown as n/a."""
    out = {}
    for label, fn in rows:
        try:
            v = fn() if callable(fn) else fn
        except Exception:
            v = "n/a"
        out[label] = v
    table(pd.Series(out, name="value").to_frame(), index=True, title=title)

def _m(res, model, met, d=3):
    """Point value of a metric with its 95% bootstrap interval, from a result with "point" and "boot"."""
    v = f"{res['point'][(model, met)]:.{d}f}"
    b = res.get("boot") or {}
    a = b.get((model, met)) if isinstance(b, dict) else None
    if a is not None and len(a) >= 20:
        lo, hi = np.nanpercentile(a, [2.5, 97.5])
        v += f" ({lo:.{d}f}-{hi:.{d}f})"
    return v

def _cv_auroc(res, model):
    """AUROC averaged over the repeats of a cross-validation result that only keeps out-of-fold predictions."""
    from sklearn.metrics import roc_auc_score
    y = res["y"]
    return float(np.mean([roc_auc_score(y, res["preds"][model][r]) for r in range(len(res["preds"][model]))]))


## S7 · Predictors and S4 / G1 — predictor table, final dictionary and reference counts

`predictors_24h_v2` has one row per obstetric hospitalization with the predictors observed in the window (admission − 24 h, admission + 24 h). `predictor_dictionary_final_v2` gives each variable its role (`main model`, `ICU sensitivity only`, `laboratory sensitivity ...`) and its plausibility limits; values outside the limits become missing. The loaded data are checked against the reference counts; the next cell shows the gate-G1 decision.

| Result | What is done | In this model |
|---|---|---|
| The three counts are reproduced | G1 passes: continue to the analyses | 19,383 deliveries, 445 SMM, 3 deaths |
| A count differs | Stop: review the tables of `sql/` before fitting any model | — |

In [ ]:
df = pd.read_sql("SELECT * FROM mimiciv.predictors_24h_v2", engine)
dic = pd.read_sql("""SELECT variable, model_role_final, missingness_indicator_main, plausible_low, plausible_high,
                            n_eligible_delivery, coverage_pct
                     FROM mimiciv.predictor_dictionary_final_v2""", engine)

for c in ["eligible_24h", "icu_in_24h", "is_delivery_v2", "smm_any", "died_same_admission"]:
    df[c] = df[c].fillna(False).astype(bool)
df["admittime"] = pd.to_datetime(df["admittime"])

VAR_MAIN = dic.loc[dic.model_role_final == "main model", "variable"].tolist()
VAR_ICU  = dic.loc[dic.model_role_final == "ICU sensitivity only", "variable"].tolist()
VAR_LAB  = dic.loc[dic.model_role_final.str.startswith("laboratory sensitivity"), "variable"].tolist()
IND_MAIN = dic.loc[dic.missingness_indicator_main.fillna(False).astype(bool), "variable"].tolist()

print("Dictionary roles ->", {"main": VAR_MAIN, "ICU only": len(VAR_ICU), "lab sensitivity": len(VAR_LAB),
                              "missingness indicator in main model": IND_MAIN})

# Plausibility limits (proposals pending clinical confirmation): out-of-range values become missing
if APPLY_LIMITS:
    n_set = {}
    for _, r in dic.dropna(subset=["plausible_low", "plausible_high"]).iterrows():
        v = r["variable"]
        if v in df.columns:
            bad = (df[v] < r["plausible_low"]) | (df[v] > r["plausible_high"])
            n_set[v] = int(bad.sum())
            df.loc[bad, v] = np.nan
    print("Values set to missing by plausibility limits (count per variable):", n_set)

# Missingness indicators (computed row by row from the raw value: no information crosses patients)
for v in set(IND_MAIN + VAR_LAB + VAR_ICU):
    df[f"{v}_missing"] = df[v].isna().astype(int)
df["icu_in_24h_bin"] = df["icu_in_24h"].astype(int)
df["y"] = df["smm_any"].astype(int)

# Audit checks against the counts already certified (aggregate only)
elig = df[df.eligible_24h & df.is_delivery_v2]
chk = {"eligible_deliveries": len(elig), "smm": int(elig.smm_any.sum()), "deaths": int(elig.died_same_admission.sum())}
for k, v in chk.items():
    print(f"{k}: {v}  (expected {EXPECTED[k]})  [{'OK' if v == EXPECTED[k] else 'CHECK: differs from the audit counts'}]")
print("Eligible any obstetric admissions:", int(df.eligible_24h.sum()),
      "| SMM:", int(df[df.eligible_24h].smm_any.sum()), "| deaths:", int(df[df.eligible_24h].died_same_admission.sum()))

In [ ]:
# Gate G1 decision: the loaded data must reproduce the reference counts frozen at the count audit (aggregates only)
g1 = pd.DataFrame([{"count": k, "loaded": v, "expected": EXPECTED[k],
                    "result": "reproduced" if v == EXPECTED[k] else "DIFFERS"} for k, v in chk.items()])
table(g1, title="Gate G1: reference counts")
table(pd.Series({"G1 decision": "passes: continue to the analyses" if (g1["result"] == "reproduced").all()
                 else "does NOT pass: review the tables of sql/ before fitting any model"}, name="decision"),
      title="Gate G1 decision")

In [ ]:
# Timing classes (one row per eligible delivery hospitalization)
tf = pd.read_sql("""SELECT hadm_id, smm_timing_class, smm_incident_strict, aki_prevalent, aki_incident
                     FROM mimiciv.timing_flags_v3""", engine)
for col in ["smm_incident_strict", "aki_prevalent", "aki_incident"]:
    tf[col] = tf[col].fillna(False).astype(bool)
df = df.drop(columns=[x for x in tf.columns if x != "hadm_id" and x in df.columns]).merge(tf, on="hadm_id", how="left")
df["smm_timing_class"] = df["smm_timing_class"].fillna("not an eligible delivery")
for col in ["smm_incident_strict", "aki_prevalent", "aki_incident"]:
    df[col] = df[col].fillna(False).astype(bool)
df["y_aki"] = df["aki_incident"].astype(int)
n_tf = int((df.eligible_24h & df.is_delivery_v2 & (df.smm_timing_class != "not an eligible delivery")).sum())
print(f"Eligible deliveries with timing flags: {n_tf} (expected {EXPECTED['eligible_deliveries']}) "
      f"[{'OK' if n_tf == EXPECTED['eligible_deliveries'] else 'CHECK'}]")

## S6 · Prediction point and window — analysis datasets and feature sets

Each analysis is a (population, feature set, label) specification. The prediction point is admission + 24 h; only information recorded up to then is used.

| Name | Population | Features |
|---|---|---|
| `main` | Eligible delivery hospitalizations | Main-model variables of the dictionary (hemoglobin, creatinine), creatinine missingness indicator, age, categorical admission variables |
| `first_admission` | Index (first) delivery hospitalization of each patient, if eligible | as `main` |
| `any_obstetric` | Eligible obstetric hospitalizations (delivery or not) | as `main` |
| `labs_extended` | as `main` | `main` + hematocrit and the other laboratory variables, each with a missingness indicator |
| `icu_vars` | as `main` | `main` + the ICU-only variables (each with a missingness indicator) + `icu_in_24h` |
| `label_ge2` | as `main` | as `main`, outcome = two or more SMM indicators (`smm_ge2`) |
| `no_icu_24h` | Eligible deliveries **without** an ICU admission in the first 24 h | as `main` |
| `icd10_only` | Eligible deliveries whose diagnoses are all ICD-10-CM (no ICD-9-CM diagnosis) | as `main` |

The table above is that of notebook 03. This notebook uses `main` (reference) and the four new specifications of the next cell: `landmark`, `incident_documented`, `incident_strict` and `aki_lab_incident`.

In [ ]:
def make_spec(name):
    d = df[df.eligible_24h & df.is_delivery_v2].copy()
    num = ["age_at_admission"] + VAR_MAIN
    binc = [f"{v}_missing" for v in IND_MAIN]
    label = "y"
    prevalent_or_indet = d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"])
    if name == "main":
        pass
    elif name in ("landmark", "incident_documented", "incident_strict"):
        d = d[~d.icu_in_24h & ~prevalent_or_indet]
        if name == "incident_documented":
            d = d[~(d.y.astype(bool) & (d.smm_timing_class == "4 untimed"))]
        if name == "incident_strict":
            d = d[~(d.y.astype(bool) & ~d.smm_incident_strict)]
    elif name == "aki_lab_incident":
        d = d[~d.icu_in_24h & ~d.aki_prevalent]
        label = "y_aki"
    else:
        raise ValueError(name)
    num  = [c for c in dict.fromkeys(num) if c in d.columns]
    binc = [c for c in dict.fromkeys(binc) if c in d.columns and d[c].nunique() > 1]
    cat  = [c for c in CAT_COLS if c in d.columns]
    d = d.reset_index(drop=True)
    return dict(name=name, data=d, num=num, bin=binc, cat=cat, label=label)

ANALYSES = ["landmark", "incident_documented", "incident_strict", "aki_lab_incident"]
for nm in ["main"] + ANALYSES:
    sp = make_spec(nm)
    d = sp["data"]
    print(f"{nm:20s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d[sp['label']].sum()):4d}  "
          f"prevalence={d[sp['label']].mean():.4f}")

## S4 · Count audit — SMM admissions by timing class

How many SMM events can be placed in time relative to T; these counts feed the cohort funnel of the article.

In [ ]:
e = df[df.eligible_24h & df.is_delivery_v2]
tab = (e[e.y == 1].groupby("smm_timing_class").size().rename("SMM admissions").to_frame())
tab["% of SMM"] = (100 * tab["SMM admissions"] / tab["SMM admissions"].sum()).round(1)
table(tab, index=True)
print(f"\nStrict incident events: {int((e.y.astype(bool) & e.smm_incident_strict).sum())}")
print(f"Laboratory AKI at or before T: {int(e.aki_prevalent.sum())};  after T: {int(e.aki_incident.sum())} "
      f"(of which with an SMM code: {int((e.aki_incident & e.y.astype(bool)).sum())})")
print(f"ICU admission before T: {int(e.icu_in_24h.sum())} admissions ({int((e.icu_in_24h & e.y.astype(bool)).sum())} with SMM)")

## S8 · Modeling, validation and metrics — preprocessing, learners, nested tuning, stacking and calibration

Everything that learns from data stays inside the training part of each split (gate G2: no patient on both sides):

* `Winsor` clips each numeric column to the 0.5th–99.5th percentiles *of the training data*.
* The median imputer, the scaler and the one-hot encoder are fitted inside a `Pipeline`, once per fit.
* Hyperparameters are chosen by **inner patient-grouped cross-validation** (average precision of the pooled out-of-fold predictions).
* The meta-learner of the ensemble is trained on the **out-of-fold** predictions of the base learners (never on in-sample predictions).
* The Platt calibrator is trained on out-of-fold predictions of the training patients; the outer test patients are only scored.

In [ ]:
def logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def expit(z):
    return 1.0 / (1.0 + np.exp(-z))

class Winsor(BaseEstimator, TransformerMixin):
    """Clip each column to training percentiles (fit on the training data only)."""
    def __init__(self, lo=0.5, hi=99.5):
        self.lo = lo; self.hi = hi
    def fit(self, X, y=None):
        X = np.asarray(X, float)
        self.lo_ = np.nanpercentile(X, self.lo, axis=0)
        self.hi_ = np.nanpercentile(X, self.hi, axis=0)
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, float), self.lo_, self.hi_)

def make_prep(num, binc, cat):
    parts = [("num", Pipeline([("w", Winsor()),
                               ("imp", SimpleImputer(strategy="median", keep_empty_features=True)),
                               ("sc", StandardScaler())]), num)]
    if binc:
        parts.append(("bin", "passthrough", binc))
    if cat:
        parts.append(("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                                       ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                                            min_frequency=0.01, sparse_output=False))]), cat))
    return ColumnTransformer(parts)

def prep_frame(d, num, binc, cat):
    X = d[num + binc + cat].copy()
    for c in cat:
        X[c] = X[c].astype(object).where(X[c].notna(), np.nan)
    return X

# Small prespecified grids (class weights are options; no resampling)
GRID = {
    # The grids cover values beyond those most often chosen in preliminary runs, so that the selected value
    # is not at the edge of the grid (ET depth, RF depth and XGB depth; class weights remain options).
    "LR":  [dict(C=C, cw=cw) for C in (0.01, 0.1, 1.0) for cw in (None, "balanced")],
    "RF":  [dict(depth=dp, cw=cw) for dp in (2, 4, 8) for cw in (None, "balanced_subsample")],
    "ET":  [dict(depth=dp, cw=cw) for dp in (4, 8, 12) for cw in (None, "balanced_subsample")],
    "XGB": [dict(depth=dp, spw=sp) for dp in (1, 2, 4) for sp in (1, "sqrt")],
}
MODELS = ["LR", "RF", "ET", "XGB", "Stack", "Stack_raw"]
BOOT_MODELS = ["LR", "RF", "ET", "XGB", "Stack"]
PAIRS = [("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "LR"), ("Stack", "XGB")]

def cfg_str(c):
    return ",".join(f"{k}={v}" for k, v in c.items())

def make_clf(name, cfg, ytr, seed):
    if name == "LR":
        return LogisticRegression(C=cfg["C"], class_weight=cfg["cw"], max_iter=3000)
    if name == "RF":
        return RandomForestClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                      class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "ET":
        return ExtraTreesClassifier(n_estimators=N_TREES, max_depth=cfg["depth"], min_samples_leaf=20,
                                    class_weight=cfg["cw"], n_jobs=N_JOBS, random_state=seed)
    if name == "XGB":
        pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
        spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
        return XGBClassifier(n_estimators=XGB_TREES, learning_rate=0.05, max_depth=cfg["depth"], subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw, eval_metric="logloss",
                             tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)
    raise ValueError(name)

def fit_predict(name, cfg, prep, Xtr, ytr, Xte, seed):
    pipe = Pipeline([("prep", clone(prep)), ("clf", make_clf(name, cfg, ytr, seed))])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

def platt(o, t, ytr):
    c = LogisticRegression(penalty=None, max_iter=1000).fit(logit(o).reshape(-1, 1), ytr)
    return c.predict_proba(logit(t).reshape(-1, 1))[:, 1]

def tune_stack_calibrate(Xtr, ytr, gtr, Xte, prep, seed):
    """Inside one outer training set: tune, build out-of-fold predictions, stack, calibrate; score the outer test set."""
    inner = list(StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed).split(Xtr, ytr, gtr))
    names = list(GRID)
    chosen, oof, pred_te = {}, {}, {}
    for name in names:
        best = None
        for cfg in GRID[name]:
            o = np.zeros(len(ytr))
            for tr, va in inner:
                o[va] = fit_predict(name, cfg, prep, Xtr.iloc[tr], ytr[tr], Xtr.iloc[va], seed)
            ap = average_precision_score(ytr, o)
            if best is None or ap > best[0]:
                best = (ap, cfg, o)
        chosen[name] = best[1]; oof[name] = best[2]
        pred_te[name] = fit_predict(name, best[1], prep, Xtr, ytr, Xte, seed)
    # explicit stacking: logistic meta-learner on out-of-fold base predictions (C = 0.1 as in the original framework)
    Z_oof = np.column_stack([logit(oof[n]) for n in names])
    Z_te = np.column_stack([logit(pred_te[n]) for n in names])
    meta = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof, ytr)
    stack_te_raw = meta.predict_proba(Z_te)[:, 1]
    # cross-fitted meta out-of-fold predictions -> calibrator never sees the predictions of its own training rows
    mo = np.zeros(len(ytr))
    for tr, va in StratifiedGroupKFold(INNER_FOLDS, shuffle=True, random_state=seed + 1).split(Z_oof, ytr, gtr):
        m = LogisticRegression(C=0.1, max_iter=2000).fit(Z_oof[tr], ytr[tr])
        mo[va] = m.predict_proba(Z_oof[va])[:, 1]
    out = {n: platt(oof[n], pred_te[n], ytr) for n in names}
    out["Stack"] = platt(mo, stack_te_raw, ytr)
    out["Stack_raw"] = stack_te_raw
    return out, chosen

def run_cv(spec, label=translate("", LANG)):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    y = d[spec["label"]].values.astype(int)
    g = d["subject_id"].values
    prep = make_prep(spec["num"], spec["bin"], spec["cat"])
    n = len(y)
    preds = {m: np.full((N_REPEATS, n), np.nan) for m in MODELS}
    chosen_log = []
    split_log = []
    t0 = time.time(); done = 0; total = N_REPEATS * OUTER_FOLDS
    for r in range(N_REPEATS):
        outer = StratifiedGroupKFold(OUTER_FOLDS, shuffle=True, random_state=SEED + r)
        for k, (tr, te) in enumerate(outer.split(X, y, g)):
            assert len(set(g[tr]) & set(g[te])) == 0, "patient present in training and test"
            split_log.append(dict(repeat=r + 1, fold=k + 1,
                                  train_adm=len(tr), train_pat=len(set(g[tr])), train_events=int(y[tr].sum()),
                                  train_event_pat=len(set(g[tr][y[tr] == 1])),
                                  test_adm=len(te), test_pat=len(set(g[te])), test_events=int(y[te].sum()),
                                  test_event_pat=len(set(g[te][y[te] == 1])), shared_patients=0))
            out, chosen = tune_stack_calibrate(X.iloc[tr], y[tr], g[tr], X.iloc[te], prep, SEED + 100 * r + k)
            for m in MODELS:
                preds[m][r, te] = out[m]
            for nme, cfg in chosen.items():
                chosen_log.append((nme, cfg_str(cfg)))
            done += 1
            if done == 1:
                el = time.time() - t0
                print(f"[{spec['name']}] first outer fold took {el/60:.1f} min -> projected total about {el*total/60:.0f} min")
    print(f"[{spec['name']}] cross-validation finished in {(time.time()-t0)/60:.1f} min; "
          f"every admission scored exactly once per repeat: {bool(np.isfinite(preds['Stack']).all())}")
    return dict(spec=spec, y=y, g=g, preds=preds, chosen=chosen_log, X=X, prep=prep, splits=pd.DataFrame(split_log))

## S8 · Modeling, validation and metrics — metrics, cluster bootstrap and paired differences

Discrimination, calibration and clinical usefulness are computed on the pooled out-of-fold predictions of each repeat and averaged over repeats. The bootstrap resamples **patients** (all admissions of a patient move together), so the intervals reflect repeated admissions. Differences between models use the **same** bootstrap samples (paired).

In [ ]:
def citl(y, z, w):
    """Calibration-in-the-large: intercept with the slope fixed at 1 (weighted Newton)."""
    a = 0.0
    for _ in range(40):
        pr = expit(z + a)
        g = np.sum(w * (y - pr)); h = np.sum(w * pr * (1 - pr))
        if h < 1e-12:
            break
        step = g / h; a += step
        if abs(step) < 1e-9:
            break
    return a

def calib_slope(y, z, w):
    """Calibration slope (weighted logistic recalibration, Newton)."""
    a, b = 0.0, 1.0
    for _ in range(40):
        pr = expit(a + b * z)
        r = w * (y - pr); v = w * pr * (1 - pr)
        gr = np.array([r.sum(), (r * z).sum()])
        H = np.array([[v.sum(), (v * z).sum()], [(v * z).sum(), (v * z * z).sum()]])
        try:
            dlt = np.linalg.solve(H, gr)
        except np.linalg.LinAlgError:
            break
        a += dlt[0]; b += dlt[1]
        if np.abs(dlt).max() < 1e-9:
            break
    return b

def top_threshold(p, w, k):
    order = np.argsort(-p)
    cw = np.cumsum(w[order])
    idx = int(np.searchsorted(cw, k * cw[-1]))
    return p[order][min(idx, len(p) - 1)]

def metrics_one(y, p, w):
    z = logit(p)
    m = {}
    m["auroc"] = roc_auc_score(y, p, sample_weight=w)
    m["auprc"] = average_precision_score(y, p, sample_weight=w)
    m["brier"] = float(np.average((p - y) ** 2, weights=w))
    m["slope"] = calib_slope(y, z, w)
    m["citl"] = citl(y, z, w)
    m["oe"] = float(np.sum(w * y) / np.sum(w * p))
    edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
    bins = np.searchsorted(edges, p)
    W = w.sum(); ece = 0.0
    for b in range(10):
        mk = bins == b; wb = w[mk].sum()
        if wb > 0:
            ece += wb / W * abs(np.sum(w[mk] * y[mk]) / wb - np.sum(w[mk] * p[mk]) / wb)
    m["ece"] = ece
    ev = np.sum(w * y)
    for k in (0.05, 0.10):
        sel = p >= top_threshold(p, w, k)
        m[f"sens_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / ev
        m[f"ppv_top{int(k*100)}"] = np.sum(w[sel] * y[sel]) / np.sum(w[sel])
    for pt in DCA_T:
        sel = p >= pt
        tp = np.sum(w[sel] * y[sel]); fp = np.sum(w[sel] * (1 - y[sel]))
        m[f"nb_{pt}"] = tp / W - fp / W * pt / (1 - pt)
    return m

def all_metrics(y, preds, w, models, strata):
    """Metrics for each model, averaged over repeats. Returns dict[(model, metric)] and the per-repeat list."""
    per = []
    for r in range(N_REPEATS):
        cur = {}
        for mname in models:
            for k, v in metrics_one(y, preds[mname][r], w).items():
                cur[(mname, k)] = v
        prev = np.sum(w * y) / w.sum()
        for pt in DCA_T:
            cur[("TreatAll", f"nb_{pt}")] = prev - (1 - prev) * pt / (1 - pt)
        for sname, mask in strata.items():
            for mname in ("LR", "Stack"):
                if mname not in models:
                    continue
                wm = w[mask]; ym = y[mask]; pm = preds[mname][r][mask]
                if np.sum(wm * ym) >= 1 and np.sum(wm * (1 - ym)) >= 1:
                    cur[(mname, f"auroc_{sname}")] = roc_auc_score(ym, pm, sample_weight=wm)
                    cur[(mname, f"auprc_{sname}")] = average_precision_score(ym, pm, sample_weight=wm)
                else:
                    cur[(mname, f"auroc_{sname}")] = np.nan
                    cur[(mname, f"auprc_{sname}")] = np.nan
        per.append(cur)
    keys = per[0].keys()
    mean = {k: float(np.nanmean([c[k] for c in per])) if not all(np.isnan(c[k]) for c in per) else np.nan for k in keys}
    return mean, per

def bootstrap(res, B, strata, seed):
    y, g, preds = res["y"], res["g"], res["preds"]
    ug, inv = np.unique(g, return_inverse=True)
    G = len(ug)
    rng = np.random.default_rng(seed)
    out = {}
    t0 = time.time()
    for b in range(B):
        cnt = np.bincount(rng.integers(0, G, G), minlength=G)
        w = cnt[inv].astype(float)
        if np.sum(w * y) < 1:
            continue
        mean, _ = all_metrics(y.astype(float), preds, w, BOOT_MODELS, strata)
        for k, v in mean.items():
            out.setdefault(k, []).append(v)
        for a_, b_ in PAIRS:            # paired differences (same bootstrap sample)
            for met in ("auroc", "auprc", "brier") + tuple(f"nb_{pt}" for pt in DCA_T):
                out.setdefault((f"{a_}-{b_}", met), []).append(mean[(a_, met)] - mean[(b_, met)])
        if b == 4:
            el = time.time() - t0
            print(f"  bootstrap: 5 replicates took {el:.0f} s -> projected {el*B/5/60:.1f} min for B={B}")
    return {k: np.array(v) for k, v in out.items()}

## S8 · S11 Reporting — report function

Builds the aggregate report of an analysis: discrimination, calibration, paired differences between models, events captured in the top 5% / 10% of predicted risk, net benefit, instability across repeats and the hyperparameters chosen in the inner loops. `emit()` saves it as `data/outputs_<analysis>.txt` (English, as cited by the article) and shows it as tables; the Spanish notebook also writes `data/outputs_<analysis>_es.txt`.

In [ ]:
def _ci(boot, key, d=3):
    a = boot.get(key)
    if a is None or len(a) < 20:
        return ""
    lo, hi = np.nanpercentile(a, [2.5, 97.5])
    return f" ({lo:.{d}f}-{hi:.{d}f})"

def decile_table(res, models=("LR", "Stack")):
    y = res["y"].astype(float)
    out = None
    for m in models:
        acc = []
        for r in range(N_REPEATS):
            p = res["preds"][m][r]
            edges = np.quantile(p, np.linspace(0, 1, 11)[1:-1])
            b = np.searchsorted(edges, p)
            acc.append(pd.DataFrame({"b": b, "p": p, "y": y}).groupby("b").agg(n=("y", "size"), pred=("p", "mean"), obs=("y", "mean")))
        mean = sum(acc) / len(acc)
        t = pd.DataFrame({"decile": mean.index + 1, "n": mean["n"].round(0).astype(int)}) if out is None else out
        t[f"{m}_pred"] = mean["pred"].round(4).values
        t[f"{m}_obs"] = mean["obs"].round(4).values
        out = t
    return out

def plot_calibration(res):
    import matplotlib.pyplot as plt
    t = decile_table(res)
    fig, ax = plt.subplots(figsize=(5, 5))
    mx = float(max(t["LR_pred"].max(), t["LR_obs"].max(), t["Stack_pred"].max(), t["Stack_obs"].max())) * 1.1
    ax.plot([0, mx], [0, mx], color="grey", lw=1, ls="--", label=translate("ideal", LANG))
    ax.plot(t["LR_pred"], t["LR_obs"], "o-", label=translate("Logistic regression", LANG))
    ax.plot(t["Stack_pred"], t["Stack_obs"], "s-", label=translate("Stacked ensemble", LANG))
    ax.set_xlabel(translate("Mean predicted risk (decile)", LANG)); ax.set_ylabel(translate("Observed SMM rate", LANG))
    ax.set_title(translate(f"Calibration by decile -- {res['spec']['name']}", LANG)); ax.legend(); plt.show()   # displayed only, not saved

def emit(name, fn):
    """Write the aggregate report data/outputs_<name>.txt (English text, as cited by the article) and show it as tables;
    the Spanish notebook also writes data/outputs_<name>_es.txt. Aggregates only."""
    path = f"../data/outputs_{name}.txt"
    text = _emit_to(path, fn)
    print(f"[aggregate report saved as data/{os.path.basename(path)}  ({len(text)} characters)]")

def run_analysis(name, B):
    spec = make_spec(name)
    res = run_cv(spec)
    d = spec["data"]
    strata = {}
    if name == "main":
        strata = {"icu1": d["icu_in_24h"].values.astype(bool), "icu0": ~d["icu_in_24h"].values.astype(bool)}
    y = res["y"].astype(float)
    res["point"], res["per"] = all_metrics(y, res["preds"], np.ones(len(y)), MODELS + [], strata)
    res["boot"] = bootstrap(res, B, strata, SEED + 7)
    res["strata"] = strata
    return res

def report(res):
    sp = res["spec"]; d = sp["data"]; y = res["y"]
    pt, per, bt = res["point"], res["per"], res["boot"]
    n, ev, npat = len(y), int(y.sum()), d.subject_id.nunique()
    print("=" * 110)
    print(f"ANALYSIS {sp['name']}: {n} admissions, {npat} patients, {ev} events ({100*ev/n:.2f}%), "
          f"numeric={len(sp['num'])}, binary={len(sp['bin'])}, categorical={len(sp['cat'])}")
    print(f"Repeats={N_REPEATS}, outer folds={OUTER_FOLDS}, inner folds={INNER_FOLDS}, bootstrap B={len(next(iter(bt.values()))) if bt else 0}, QUICK={QUICK}")
    print("=" * 110)

    sl = res["splits"]
    print("\n[0] Patients per outer partition (repeat 1 shown; every split verified to share no patient between training and test)")
    table(sl[sl.repeat == 1].drop(columns=["repeat"]))
    print(f"    Across all {len(sl)} splits: test patients per fold {sl.test_pat.min()}-{sl.test_pat.max()}, "
          f"event-bearing test patients per fold {sl.test_event_pat.min()}-{sl.test_event_pat.max()}, "
          f"patients present on both sides: {int(sl.shared_patients.sum())}")

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        r["sd_auroc_repeats"] = f"{np.std([c[(m,'auroc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        r["sd_auprc_repeats"] = f"{np.std([c[(m,'auprc')] for c in per], ddof=1):.4f}" if len(per) > 1 else "n/a"
        rows.append(r)
    print("\n[1] Discrimination and overall accuracy (mean over repeats; 95% cluster-bootstrap CI)")
    print(f"    prevalence = {y.mean():.4f}  (AUPRC of a non-informative model equals the prevalence)")
    table(pd.DataFrame(rows))

    rows = []
    for m in MODELS:
        r = {"model": m}
        for met, dd in [("slope", 2), ("citl", 2), ("oe", 2), ("ece", 4)]:
            r[met] = f"{pt[(m, met)]:.{dd}f}" + (_ci(bt, (m, met), dd) if m in BOOT_MODELS else "")
        rows.append(r)
    print("\n[2] Calibration (slope ideal 1; calibration-in-the-large ideal 0; O/E ideal 1; ECE with 10 quantile bins)")
    table(pd.DataFrame(rows))

    rows = []
    for a_, b_ in PAIRS:
        r = {"comparison (first minus second)": f"{a_} - {b_}"}
        for met, dd in [("auroc", 3), ("auprc", 3), ("brier", 5)]:
            r[met] = f"{pt[(a_, met)] - pt[(b_, met)]:+.{dd}f}" + _ci(bt, (f"{a_}-{b_}", met), dd)
        rows.append(r)
    print("\n[3] Paired differences (same bootstrap samples); the last row compares the ensemble with XGBoost")
    table(pd.DataFrame(rows))

    rows = []
    for m in BOOT_MODELS:
        rows.append({"model": m,
                     "sens_top5%": f"{pt[(m,'sens_top5')]:.3f}" + _ci(bt, (m, "sens_top5")),
                     "ppv_top5%": f"{pt[(m,'ppv_top5')]:.3f}",
                     "sens_top10%": f"{pt[(m,'sens_top10')]:.3f}" + _ci(bt, (m, "sens_top10")),
                     "ppv_top10%": f"{pt[(m,'ppv_top10')]:.3f}"})
    print("\n[4] Events captured among the 5% / 10% of admissions with the highest predicted risk")
    table(pd.DataFrame(rows))

    rows = []
    for m in BOOT_MODELS + ["TreatAll"]:
        rows.append({"strategy": m, **{f"pt={t:g}": round(pt[(m, f'nb_{t}')], 5) for t in DCA_T}})
    rows.append({"strategy": "TreatNone", **{f"pt={t:g}": 0.0 for t in DCA_T}})
    print("\n[5] Net benefit (decision curve) at the stated threshold probabilities")
    table(pd.DataFrame(rows))
    rows = []
    for a_, b_ in [("Stack", "LR"), ("RF", "LR"), ("ET", "LR"), ("XGB", "LR"), ("Stack", "XGB")]:
        rows.append({"comparison": f"{a_} - {b_}", **{f"pt={t:g}": f"{pt[(a_,f'nb_{t}')]-pt[(b_,f'nb_{t}')]:+.5f}" + _ci(bt, (f"{a_}-{b_}", f"nb_{t}"), 5) for t in DCA_T[:6]}})
    print("    Paired net-benefit differences:")
    table(pd.DataFrame(rows))

    if res["strata"]:
        rows = []
        for sname, lab in [("icu1", "ICU admission within 24 h"), ("icu0", "no ICU within 24 h")]:
            mk = res["strata"][sname]
            r = {"stratum": lab, "n": int(mk.sum()), "events": int(y[mk].sum())}
            for m in ("LR", "Stack"):
                r[f"{m}_auroc"] = (f"{pt[(m, 'auroc_' + sname)]:.3f}" + _ci(bt, (m, "auroc_" + sname))) if not np.isnan(pt[(m, 'auroc_' + sname)]) else "n/a"
                r[f"{m}_auprc"] = (f"{pt[(m, 'auprc_' + sname)]:.3f}") if not np.isnan(pt[(m, 'auprc_' + sname)]) else "n/a"
            rows.append(r)
        print("\n[6] Performance by ICU admission within the first 24 h (same predictions, stratified evaluation)")
        table(pd.DataFrame(rows))

    print("\n[8] Calibration by decile of predicted risk (outer-test predictions; mean over repeats)")
    table(decile_table(res))

    ch = pd.DataFrame(res["chosen"], columns=["learner", "config"])
    tab = ch.value_counts().rename("outer_fits").reset_index().sort_values(["learner", "outer_fits"], ascending=[True, False])
    print(f"\n[7] Hyperparameters selected in the inner loops (out of {N_REPEATS*OUTER_FOLDS} outer fits per learner)")
    table(tab)

## S9 · C1 · C2 — landmark and incident populations

`landmark` serves criterion C1; `incident_documented`, `incident_strict` and `aki_lab_incident` serve C2. Analyses with fewer than 40 events are run but flagged as unstable. `RUN_MAIN_ANALYSES = False` skips them.

| Result | What is done | In this model |
|---|---|---|
| AUROC drop ≤ 0.05 in the landmark population (C1) | C1 met | Decided on the fitted model in notebook 10 |
| AUROC ≥ 0.75 with lower limit ≥ 0.70 for a timed outcome (C2) | C2 met | Decided in notebook 10 |

In [ ]:

RUN_MAIN_ANALYSES = True   # set to False to run only section 8 without repeating the four analyses above
RUN_MAIN_ANALYSES = False 
results = {}
if RUN_MAIN_ANALYSES:
    for nm in ANALYSES:
        sp = make_spec(nm)
        ev = int(sp["data"][sp["label"]].sum())
        if ev < 40:
            print(f"WARNING: {nm} has only {ev} events; estimates will be unstable (report as exploratory).")
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

## S9 · C3 — detection bias of laboratory AKI and shared variable

Laboratory AKI after T can only be detected if creatinine is measured after T, and the predictor (creatinine up to 24 h) and the outcome (later creatinine) are the same variable.

| Analysis | Population | Predictors |
|---|---|---|
| `aki_lab_incident_tested` | `aki_lab_incident` restricted to admissions **with creatinine measured in the first 24 h** | as `main` |
| `aki_lab_incident_tested_nocreat` | same | as `main` **without the creatinine value** |

| Result | What is done | In this model |
|---|---|---|
| AUROC falls by 0.05 or less among the tested | C3 met | — |
| AUROC falls by more than 0.05 | C3 not met: detection bias | 0.850 in all, 0.628 among the tested |

In [ ]:
CREAT = [v for v in VAR_MAIN if "creat" in v.lower()]
assert len(CREAT) == 1, f"creatinine variable not found in VAR_MAIN: {VAR_MAIN}"
CREAT = CREAT[0]
_make_spec_base = make_spec

def make_spec(name):
    if name not in ("aki_lab_incident_tested", "aki_lab_incident_tested_nocreat"):
        return _make_spec_base(name)
    sp = _make_spec_base("aki_lab_incident")
    d = sp["data"]
    d = d[d[CREAT].notna()].reset_index(drop=True)
    num = [v for v in sp["num"] if not (name.endswith("_nocreat") and v == CREAT)]
    binc = [b for b in sp["bin"] if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=sp["label"])

SENS_AKI = ["aki_lab_incident_tested", "aki_lab_incident_tested_nocreat"]
base = _make_spec_base("aki_lab_incident")["data"]
print(f"aki_lab_incident population: {len(base)} admissions, {int(base.y_aki.sum())} events "
      f"({100 * base.y_aki.mean():.2f}%)")
tested = base[CREAT].notna()
print(f"  creatinine measured in the first 24 h: {int(tested.sum())} admissions ({100 * tested.mean():.1f}%), "
      f"{int(base.loc[tested, 'y_aki'].sum())} events ({100 * base.loc[tested, 'y_aki'].mean():.2f}%)")
print(f"  not measured:                          {int((~tested).sum())} admissions, "
      f"{int(base.loc[~tested, 'y_aki'].sum())} events ({100 * base.loc[~tested, 'y_aki'].mean():.2f}%)")
for nm in SENS_AKI:
    sp = make_spec(nm)
    print(f"{nm:34s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  numeric={sp['num']}  binary={sp['bin']}")

RUN_SECTION_8 = True   # set to False to skip these two analyses when running section 9 only
if RUN_SECTION_8:
    for nm in SENS_AKI:
        ev = int(make_spec(nm)["data"]["y_aki"].sum())
        if ev < 40:
            print(f"WARNING: {nm} has only {ev} events; estimates will be unstable (report as exploratory).")
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

## S9 · S3 — SMM definition without the admissions that are SMM only because of non-acute conditions

The CDC list includes a few codes for chronic or incidental conditions (nonruptured cerebral aneurysm, carotid stenosis, amaurosis fugax, thoracic aortic aneurysm without rupture). The code audit (`sql/07_timing_audit/02_phenotype_audit_v3.sql`, query A2) found 14 SMM admissions whose only codes are of this kind; they are removed (not relabelled) in `excl_nonacute` and `landmark_excl_nonacute`. Requires `mimiciv.smm_code_hits_v3`. `RUN_SECTION_9 = False` skips this section.

In [ ]:
from sqlalchemy import text
NON_ACUTE_CODES = ("I671", "4373", "I6521", "I6522", "G453", "I712")
codes_sql = ", ".join(f"'{x}'" for x in NON_ACUTE_CODES)   # fixed constants, not user input
SQL_NON_ACUTE = text(f"""
    SELECT hadm_id
    FROM mimiciv.smm_code_hits_v3
    GROUP BY hadm_id
    HAVING BOOL_AND(icd_code IN ({codes_sql}))
""")
non_acute = set(pd.read_sql(SQL_NON_ACUTE, engine).hadm_id)   # kept in memory only

if "_make_spec_s8" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s8 = make_spec
def make_spec(name):
    if name not in ("excl_nonacute", "landmark_excl_nonacute"):
        return _make_spec_s8(name)
    sp = _make_spec_s8("main" if name == "excl_nonacute" else "landmark")
    d = sp["data"]
    d = d[~d.hadm_id.isin(non_acute)].reset_index(drop=True)
    return dict(name=name, data=d, num=sp["num"], bin=sp["bin"], cat=sp["cat"], label=sp["label"])

SENS_NONACUTE = ["excl_nonacute", "landmark_excl_nonacute"]
elig = df[df.eligible_24h & df.is_delivery_v2]
print(f"Admissions that are SMM only because of non-acute codes: {len(non_acute)} (expected 14 from query A2)  "
      f"[{'OK' if len(non_acute) == 14 else 'CHECK'}]")
print(f"  of which eligible SMM deliveries: {int(elig[elig.hadm_id.isin(non_acute)].y.sum())}")
for ref, nm in (("main", "excl_nonacute"), ("landmark", "landmark_excl_nonacute")):
    a, b = make_spec(ref)["data"], make_spec(nm)["data"]
    print(f"{nm:24s} admissions={len(b)} (reference {ref}: {len(a)})  events={int(b.y.sum())} (reference: {int(a.y.sum())})")

RUN_SECTION_9 = True
if RUN_SECTION_9:
    for nm in SENS_NONACUTE:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

## R4 → S7 · C1 · C2 — do richer predictors improve the prediction of SMM not yet present at 24 h?

With the full label, the other laboratory tests raised the AUROC from 0.79 to 0.82; this section repeats the comparison where SMM present at T has been removed. `landmark_labs_noind` separates the information in the **values** from the information in **which tests were ordered** (the indicators).

| Analysis | Population | Predictors |
|---|---|---|
| `landmark_labs` | `landmark` | main + all `laboratory sensitivity` tests, each with a missingness indicator |
| `landmark_labs_noind` | `landmark` | as `landmark_labs` without missingness indicators |
| `incident_documented_labs` | `incident_documented` | as `landmark_labs` |
| `incident_strict_labs` | `incident_strict` | as `landmark_labs` |

| Result | What is done | In this model |
|---|---|---|
| The gain survives in the landmark population and without indicators | R4 may turn the model into a predictor: reclassify at G4 | — |
| The gain comes from which tests were ordered | The model stays an early identifier | Gain mostly from the indicators |

In [ ]:
if "_make_spec_s9" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s9 = make_spec
LAB10 = ["landmark_labs", "landmark_labs_noind", "incident_documented_labs", "incident_strict_labs"]

def make_spec(name):
    if name not in LAB10:
        return _make_spec_s9(name)
    base_name = name.replace("_labs_noind", "").replace("_labs", "")
    sp = _make_spec_s9(base_name)
    d = sp["data"]
    num = list(dict.fromkeys(sp["num"] + [v for v in VAR_LAB if v in d.columns]))
    if name.endswith("_noind"):
        binc = []
    else:
        binc = list(dict.fromkeys(sp["bin"] + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
        binc = [b for b in binc if d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=sp["label"])

print("Laboratory variables added:", VAR_LAB)
for nm in LAB10:
    sp = make_spec(nm)
    print(f"{nm:26s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

RUN_SECTION_10 = True
if RUN_SECTION_10:
    for nm in LAB10:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def richer_summary():
    rows = []
    for nm in LAB10:
        if nm not in results:
            continue
        pt, bt, y = results[nm]["point"], results[nm]["boot"], results[nm]["y"]
        for m in ("LR", "XGB", "Stack"):
            rows.append({"analysis": nm, "model": m, "events": int(y.sum()), "prevalence": f"{y.mean():.4f}",
                         "auroc": f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc")),
                         "auprc": f"{pt[(m, 'auprc')]:.3f}" + _ci(bt, (m, "auprc")),
                         "slope": f"{pt[(m, 'slope')]:.2f}", "O/E": f"{pt[(m, 'oe')]:.2f}"})
    print(f"RICHER PREDICTORS SUMMARY (QUICK={QUICK}); references: outputs_landmark, outputs_incident_documented, outputs_incident_strict")
    table(pd.DataFrame(rows))

if RUN_SECTION_10:
    emit("richer_predictors_summary", richer_summary)

## R2 → S3 · C2 — escalation of care after the prediction point

A timed outcome that does not depend on laboratory testing: ICU admission, start of invasive ventilation, a procedure-based SMM indicator or in-hospital death **after T** (`sql/07_timing_audit/04_escalation_label.sql`, table `mimiciv.escalation_v3`). Admissions with ICU before T are excluded (expected: 19,268 admissions, 101 events).

| Analysis | Population | Predictors |
|---|---|---|
| `escalation` | eligible deliveries without ICU before T | main model |
| `escalation_labs` | same | main + all `laboratory sensitivity` tests with indicators (R2 combined with R4) |
| `escalation_landmark` | as `escalation`, also without SMM present or indeterminate at T | main model |

| Result | What is done | In this model |
|---|---|---|
| AUROC ≥ 0.75 and lower 95% limit ≥ 0.70 | C2 met for a timed outcome; check G3 (section 14) | 0.763 (0.709–0.808); landmark 0.765 |
| Fewer than 100 events | Report as exploratory | 101 events |

In [ ]:
EXPECTED_ESC = {"population": 19268, "escalation": 101}   # Q-E1 of sql/07_timing_audit/03_escalation_counts.sql
esc = pd.read_sql("""SELECT hadm_id, icu_after, vent_after, smm_proc_after, death_after, escalation_any
                     FROM mimiciv.escalation_v3""", engine)
for c in ["icu_after", "vent_after", "smm_proc_after", "death_after", "escalation_any"]:
    esc[c] = esc[c].fillna(False).astype(bool)
chk = {"population": len(esc), "escalation": int(esc.escalation_any.sum())}
for k, v in chk.items():
    print(f"escalation {k}: {v}  (expected {EXPECTED_ESC[k]})  [{'OK' if v == EXPECTED_ESC[k] else 'CHECK: differs from Q-E1'}]")

df = df.drop(columns=[c for c in esc.columns if c != "hadm_id" and c in df.columns])
df = df.merge(esc, on="hadm_id", how="left")
df["in_esc_pop"] = df["escalation_any"].notna()
df["y_esc"] = df["escalation_any"].fillna(False).astype(int)

def _n(x):
    return str(int(x)) if x >= 10 else "<10"

if "_make_spec_s10b" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s10b = make_spec
ESC11 = ["escalation", "escalation_labs", "escalation_landmark"]

def make_spec(name):
    if name not in ESC11:
        return _make_spec_s10b(name)
    sp = _make_spec_s10b("main")
    d = sp["data"]
    d = d[d.in_esc_pop & ~d.icu_in_24h]
    if name == "escalation_landmark":
        d = d[~d.smm_timing_class.isin(["1 prevalent", "2 indeterminate"])]
    d = d.reset_index(drop=True)
    num, binc = list(sp["num"]), list(sp["bin"])
    if name == "escalation_labs":
        num = list(dict.fromkeys(num + [v for v in VAR_LAB if v in d.columns]))
        binc = list(dict.fromkeys(binc + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
    binc = [b for b in binc if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=translate("y_esc", LANG))

for nm in ESC11:
    sp = make_spec(nm); d = sp["data"]
    print(f"{nm:20s} admissions={len(d):6d}  patients={d.subject_id.nunique():6d}  events={int(d.y_esc.sum()):4d}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

RUN_SECTION_11 = True
RUN_SECTION_11 = False

if RUN_SECTION_11:
    for nm in ESC11:
        ev = int(make_spec(nm)["data"]["y_esc"].sum())
        if ev < 100:
            print(f"NOTE: {nm} has {ev} events (< 100): report as exploratory.")
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def escalation_summary():
    print(f"ESCALATION AFTER T (RETURN R2) SUMMARY (QUICK={QUICK}); bootstrap B={BOOT_SENS}; counts below 10 shown as <10")
    print("\n[1] Components of the composite (an admission can have several)")
    rows = []
    for nm in ESC11:
        d = make_spec(nm)["data"]; ev = d[d.y_esc == 1]
        rows.append({"analysis": nm, "admissions": _n(len(d)), "escalation": _n(len(ev)),
                     "icu_after": _n(ev.icu_after.sum()), "vent_after": _n(ev.vent_after.sum()),
                     "smm_proc_after": _n(ev.smm_proc_after.sum()), "death_after": _n(ev.death_after.sum()),
                     "with_smm_label": _n(ev.y.sum())})
    table(pd.DataFrame(rows))
    print("\n[2] Discrimination and calibration; C2 = AUROC >= 0.75 and lower 95% limit >= 0.70")
    rows = []
    for nm in ESC11:
        if nm not in results:
            continue
        pt, bt, y = results[nm]["point"], results[nm]["boot"], results[nm]["y"]
        for m in ("LR", "XGB", "Stack"):
            lo = np.nanpercentile(bt[(m, "auroc")], 2.5) if (m, "auroc") in bt else np.nan
            rows.append({"analysis": nm, "model": m, "events": int(y.sum()), "prevalence": f"{y.mean():.4f}",
                         "auroc": f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc")),
                         "auprc": f"{pt[(m, 'auprc')]:.3f}" + _ci(bt, (m, "auprc")),
                         "slope": f"{pt[(m, 'slope')]:.2f}", "O/E": f"{pt[(m, 'oe')]:.2f}",
                         "sens_top10": f"{pt[(m, 'sens_top10')]:.3f}",
                         "C2": "met" if (pt[(m, 'auroc')] >= 0.75 and lo >= 0.70) else "not met"})
    table(pd.DataFrame(rows))

if RUN_SECTION_11:
    emit("escalation_summary", escalation_summary)

## S7 · S10 — race and ethnicity: model without the variable and performance by group

SMM rates differ by race and ethnicity, so omitting the variable may miscalibrate the model for the groups at highest risk; but race in clinical algorithms can also reproduce inequities. `main_norace` and `landmark_norace` remove `ethnicity`; `race_subgroups` reports admissions, events, AUROC, AUPRC, calibration-in-the-large, slope and O/E by group (White, Black, Hispanic/Latina, Asian, Other, Unknown or declined), with counts below 10 shown as `<10`.

| Result | What is done | In this model |
|---|---|---|
| Removing race does not change discrimination but worsens calibration in some groups | Keep race in the model and report by group | Calibration worse in the Hispanic and Asian groups without race |
| A group's discrimination is clearly lower in the landmark population | Explicit caution for any implementation | Black 0.689 vs White 0.741 |

In [ ]:
RACE_COL = "ethnicity"
MIN_EVENTS_GROUP = 20
SUB_MODELS = ("LR", "XGB", "Stack")

def race_group(v):
    s = str(v).upper().strip() if pd.notna(v) else "UNKNOWN"
    if s.startswith("WHITE"):
        return "White"
    if s.startswith("BLACK"):
        return "Black"
    if s.startswith("HISPANIC") or s.startswith("SOUTH AMERICAN"):
        return "Hispanic/Latina"
    if s.startswith("ASIAN"):
        return "Asian"
    if s in ("UNKNOWN", "UNABLE TO OBTAIN", "PATIENT DECLINED TO ANSWER", "NONE", ""):
        return "Unknown or declined"
    return "Other"

def _n(x):
    return str(int(x)) if x >= 10 else "<10"

if "_make_spec_s10" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s10 = make_spec
NORACE = ["main_norace", "landmark_norace"]

def make_spec(name):
    if name not in NORACE:
        return _make_spec_s10(name)
    sp = _make_spec_s10(name.replace("_norace", ""))
    assert RACE_COL in sp["cat"], f"{RACE_COL} not among the categorical predictors: {sp['cat']}"
    return dict(sp, name=name, cat=[c for c in sp["cat"] if c != RACE_COL])

# Mapping of the recorded categories to the collapsed groups (eligible deliveries; aggregates only)
e = df[df.eligible_24h & df.is_delivery_v2]
mp = (e.assign(group=e[RACE_COL].map(race_group))
        .groupby(["group", RACE_COL], dropna=False).agg(admissions=("y", "size"), smm=("y", "sum")).reset_index())
mp["admissions"] = mp["admissions"].map(_n); mp["smm"] = mp["smm"].map(_n)
print("Recorded race and ethnicity categories and the collapsed group (counts below 10 suppressed):")
table(mp)
for nm in NORACE:
    sp = make_spec(nm)
    print(f"{nm:18s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  categorical={sp['cat']}")

def group_metrics(res, B, seed):
    """Metrics by collapsed race group, averaged over repeats; patient bootstrap within each group."""
    d = res["spec"]["data"]; y = res["y"].astype(float); g = res["g"]
    grp = d[RACE_COL].map(race_group).values
    rows, rng = [], np.random.default_rng(seed)
    for gr in ["White", "Black", "Hispanic/Latina", "Asian", "Other", "Unknown or declined"]:
        mk = grp == gr
        n, ev = int(mk.sum()), int(y[mk].sum())
        if n == 0:
            continue
        base = {"analysis": res["spec"]["name"], "group": gr, "admissions": _n(n), "events": _n(ev)}
        if ev < MIN_EVENTS_GROUP or (n - ev) < MIN_EVENTS_GROUP:
            rows.append(dict(base, model="-", note=f"fewer than {MIN_EVENTS_GROUP} events or non-events: counts only"))
            continue
        yg, gg = y[mk], g[mk]
        ug, inv = np.unique(gg, return_inverse=True)
        point = {m: pd.DataFrame([metrics_one(yg, res["preds"][m][r][mk], np.ones(n)) for r in range(N_REPEATS)]).mean()
                 for m in SUB_MODELS}
        boot = {m: {"auroc": [], "oe": []} for m in SUB_MODELS}
        for b in range(B):
            w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)
            if np.sum(w * yg) < 1 or np.sum(w * (1 - yg)) < 1:
                continue
            for m in SUB_MODELS:
                a, o = [], []
                for r in range(N_REPEATS):
                    pr = res["preds"][m][r][mk]
                    a.append(roc_auc_score(yg, pr, sample_weight=w))
                    o.append(np.sum(w * yg) / np.sum(w * pr))
                boot[m]["auroc"].append(np.mean(a)); boot[m]["oe"].append(np.mean(o))
        for m in SUB_MODELS:
            pt = point[m]
            ci = lambda k, dd: " ({:.{d}f}-{:.{d}f})".format(*np.nanpercentile(boot[m][k], [2.5, 97.5]), d=dd)
            rows.append(dict(base, model=m, prevalence=f"{ev / n:.4f}",
                             auroc=f"{pt['auroc']:.3f}" + ci("auroc", 3), auprc=f"{pt['auprc']:.3f}",
                             citl=f"{pt['citl']:.2f}", slope=f"{pt['slope']:.2f}",
                             oe=f"{pt['oe']:.2f}" + ci("oe", 2), note=""))
    return rows

def paired_race_diff(res_a, res_b, B, seed):
    """Overall AUROC/AUPRC with race (a) minus without race (b), same patient bootstrap samples."""
    assert np.array_equal(res_a["y"], res_b["y"]) and np.array_equal(res_a["g"], res_b["g"]), "rows not aligned"
    y = res_a["y"].astype(float); ug, inv = np.unique(res_a["g"], return_inverse=True)
    rng = np.random.default_rng(seed); rows = []
    for m in SUB_MODELS:
        dif = {"auroc": [], "auprc": []}
        for b in range(B):
            w = np.bincount(rng.integers(0, len(ug), len(ug)), minlength=len(ug))[inv].astype(float)
            if np.sum(w * y) < 1:
                continue
            for met, fn in (("auroc", roc_auc_score), ("auprc", average_precision_score)):
                va = np.mean([fn(y, res_a["preds"][m][r], sample_weight=w) for r in range(N_REPEATS)])
                vb = np.mean([fn(y, res_b["preds"][m][r], sample_weight=w) for r in range(N_REPEATS)])
                dif[met].append(va - vb)
        r0 = {"comparison": f"{res_a['spec']['name']} - {res_b['spec']['name']}", "model": m}
        for met in ("auroc", "auprc"):
            pa = res_a["point"][(m, met)]; pb = res_b["point"][(m, met)]
            lo, hi = np.nanpercentile(dif[met], [2.5, 97.5])
            r0[met] = f"{pa - pb:+.3f} ({lo:+.3f} to {hi:+.3f})"
        rows.append(r0)
    return rows

RUN_SECTION_12 = True
if RUN_SECTION_12:
    for nm in ["main", "landmark"]:
        if nm not in results:
            results[nm] = run_analysis(nm, BOOT_SENS)
            emit(nm + "_rerun12", lambda r=results[nm]: report(r))
    for nm in NORACE:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def race_summary():
    print(f"RACE AND ETHNICITY SUMMARY (QUICK={QUICK}); bootstrap B={BOOT_SENS}; groups with < {MIN_EVENTS_GROUP} events or non-events: counts only")
    print("\n[1] Overall discrimination, with race minus without race (paired patient bootstrap)")
    rows = []
    for a, b in (("main", "main_norace"), ("landmark", "landmark_norace")):
        if a in results and b in results:
            rows += paired_race_diff(results[a], results[b], BOOT_SENS, SEED + 120)
    table(pd.DataFrame(rows))
    print("\n[2] Performance and calibration by racial and ethnic group")
    rows = []
    for nm in ("main", "main_norace", "landmark", "landmark_norace"):
        if nm in results:
            rows += group_metrics(results[nm], BOOT_SENS, SEED + 121)
    table(pd.DataFrame(rows).fillna(""))

if RUN_SECTION_12:
    emit("race_subgroups", race_summary)

## R4 · C3 — detection bias for the extended laboratory model

Same test as section 8 for the model of return R4. C3 is met if the AUROC falls by 0.05 or less from `aki_lab_incident_labs` to `aki_lab_incident_tested_labs` (admissions with creatinine measured in the first 24 h).

In [ ]:
if "_make_spec_s12" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s12 = make_spec
C3LABS = ["aki_lab_incident_labs", "aki_lab_incident_tested_labs"]

def make_spec(name):
    if name not in C3LABS:
        return _make_spec_s12(name)
    sp = _make_spec_s12("aki_lab_incident")
    d = sp["data"]
    if name == "aki_lab_incident_tested_labs":
        d = d[d[CREAT].notna()]
    d = d.reset_index(drop=True)
    num = list(dict.fromkeys(sp["num"] + [v for v in VAR_LAB if v in d.columns]))
    binc = list(dict.fromkeys(sp["bin"] + [f"{v}_missing" for v in VAR_LAB if f"{v}_missing" in d.columns]))
    binc = [b for b in binc if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=num, bin=binc, cat=sp["cat"], label=sp["label"])

for nm in C3LABS:
    sp = make_spec(nm)
    print(f"{nm:30s} admissions={len(sp['data'])}  events={int(sp['data'][sp['label']].sum())}  "
          f"numeric={len(sp['num'])}  binary={len(sp['bin'])}  categorical={len(sp['cat'])}")

RUN_SECTION_13 = True
if RUN_SECTION_13:
    for nm in C3LABS:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def c3_labs_summary():
    print(f"CRITERION C3 FOR THE EXTENDED LABORATORY MODEL (QUICK={QUICK}); bootstrap B={BOOT_SENS}")
    rows = []
    for m in ("LR", "XGB", "Stack"):
        r = {"model": m}
        for nm in C3LABS:
            if nm in results:
                pt, bt = results[nm]["point"], results[nm]["boot"]
                r[f"{nm} events"] = int(results[nm]["y"].sum())
                r[f"{nm} auroc"] = f"{pt[(m, 'auroc')]:.3f}" + _ci(bt, (m, "auroc"))
        if all(nm in results for nm in C3LABS):
            drop = results[C3LABS[0]]["point"][(m, "auroc")] - results[C3LABS[1]]["point"][(m, "auroc")]
            r["auroc drop"] = f"{drop:+.3f}"
            r["C3"] = "met" if drop <= 0.05 else "not met"
        rows.append(r)
    table(pd.DataFrame(rows))

if RUN_SECTION_13:
    emit("c3_labs_summary", c3_labs_summary)

## R2 · G3 — sample size for the outcome of return R2 (escalation of care)

Return R2 sends the process back to S3 with a new outcome, so the new outcome must pass gate G3 again. Riley criteria with the Cox-Snell R² of the cross-validated logistic regression; the main SMM analysis is added as a check (9,112 admissions and 209 events required). If the out-of-fold predictions are not in memory, the cross-validation is re-run without bootstrap.

| Result | What is done | In this model |
|---|---|---|
| Events available ≥ required | G3 passes | SMM main model |
| Events available < required | Sample-size reduction (section 15) | Escalation: 101 events, 194 required with 32 parameters |

In [ ]:
def riley(P, phi, r2cs, shrink=0.9, deltas=(0.05, 0.01)):
    lnl0 = phi * math.log(phi) + (1 - phi) * math.log(1 - phi)
    max_r2 = 1 - math.exp(2 * lnl0)
    n1 = P / ((shrink - 1) * math.log(1 - r2cs / shrink))
    s_vh = r2cs / (r2cs + 0.05 * max_r2)
    n2 = P / ((s_vh - 1) * math.log(1 - r2cs / s_vh))
    out = {"max_r2cs": max_r2, "n_shrinkage": n1, "n_optimism": n2}
    for dl in deltas:
        out[f"n_risk_delta{dl}"] = (1.96 / dl) ** 2 * phi * (1 - phi)
    n = max(n1, n2, out[f"n_risk_delta{deltas[-1]}"])
    out["n_required"] = n
    out["events_required"] = n * phi
    out["epp_at_required"] = n * phi / P
    out["P_max_supported"] = None
    return out

def p_max(n, r2cs, shrink=0.9):
    """Largest number of parameters with expected shrinkage >= 0.9 for a given n and R2 (criterion 1 solved for P)."""
    return n * (shrink - 1) * math.log(1 - r2cs / shrink)

def riley_block(label, n, events, P, r2_list):
    phi = events / n
    print(f"\n{label}: n={n}, events={events}, prevalence={phi:.5f}, candidate parameters P={P}, EPP={events/P:.2f}")
    rows = []
    for tag, r2 in r2_list:
        rr = riley(P, phi, r2)
        rows.append({"R2_CS scenario": tag, "R2_CS": f"{r2:.4f}", "max R2_CS": f"{rr['max_r2cs']:.4f}",
                     "n shrinkage": f"{rr['n_shrinkage']:.0f}", "n optimism": f"{rr['n_optimism']:.0f}",
                     "n risk(+-0.05)": f"{rr['n_risk_delta0.05']:.0f}", "n risk(+-0.01)": f"{rr['n_risk_delta0.01']:.0f}",
                     "n required": f"{rr['n_required']:.0f}", "events required": f"{rr['events_required']:.0f}",
                     "available n": n, "available events": events,
                     "max P supported (n,R2)": f"{p_max(n, r2):.1f}"})
    table(pd.DataFrame(rows))

def count_parameters(spec):
    d = spec["data"]
    X = prep_frame(d, spec["num"], spec["bin"], spec["cat"])
    return int(make_prep(spec["num"], spec["bin"], spec["cat"]).fit(X).transform(X).shape[1])

def r2cs_from_oof(res):
    y = res["y"].astype(float); n = len(y); phi = y.mean()
    lnl0 = n * (phi * math.log(phi) + (1 - phi) * math.log(1 - phi))
    vals = []
    for r in range(N_REPEATS):
        p = np.clip(res["preds"]["LR"][r], 1e-9, 1 - 1e-9)
        lnl1 = float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))
        vals.append(max(1 - math.exp(2 / n * (lnl0 - lnl1)), 1e-6))
    return float(np.mean(vals))
P3_ANALYSES = ["main", "escalation", "escalation_labs", "escalation_landmark"]

RUN_SECTION_14 = True
RUN_SECTION_14 = False

p3_res = {}
if RUN_SECTION_14:
    for nm in P3_ANALYSES:
        if nm in results:
            p3_res[nm] = results[nm]
        else:
            print(f"[{nm}] not in memory: re-running the cross-validation without bootstrap")
            p3_res[nm] = run_cv(make_spec(nm))

def p3_report():
    print(f"GATE G3 (RILEY CRITERIA) FOR THE OUTCOME OF RETURN R2 (QUICK={QUICK})")
    print("R2_CS observed = mean over repeats of the Cox-Snell R2 of the out-of-fold logistic regression predictions")
    for nm in P3_ANALYSES:
        if nm not in p3_res:
            continue
        res = p3_res[nm]; sp = res["spec"]
        n, ev = len(res["y"]), int(res["y"].sum())
        P = count_parameters(sp)
        r2 = r2cs_from_oof(res)
        riley_block(f"{nm} (label {sp['label']})", n, ev, P, [("observed (cross-validated LR)", r2)])
        verdict = "passes" if riley(P, ev / n, r2)["n_required"] <= n else "does NOT pass"
        print(f"  -> gate G3 {verdict}: {n} admissions available")

if RUN_SECTION_14:
    emit("g3_escalation", p3_report)


## G3 · sample-size reduction for the outcome of return R2

Rule recorded in `docs/decision_G3_reduction_2026-11-09.md` before this section was run: a model that meets every applicable G4 criterion but fails G3 is refitted **once** with a prespecified reduced predictor set (at most the parameters the data support, chosen without looking at the outcome), then checked again at G3 and at C1 and C2. Its C4 is in notebook 09, section 10.

| Result | What is done | In this model |
|---|---|---|
| G3 passes and C1, C2 met | Exploratory signal of prediction | Passes G3 (89 events required); landmark 0.752; XGBoost 0.749 |
| G3 fails again | Feasibility only | — |

In [ ]:
def add_reduced_columns(frame):
    """Columns of the prespecified reduced set (docs/decision_G3_reduction_2026-11-09.md). Row-wise, no outcome information."""
    eth = frame["ethnicity"].fillna("").astype(str).str.upper()
    frame["race5"] = np.select(
        [eth.str.startswith("WHITE"), eth.str.startswith("BLACK"),
         eth.str.startswith("HISPANIC") | eth.str.startswith("SOUTH AMERICAN"), eth.str.startswith("ASIAN")],
        ["White", "Black", "Hispanic or Latina", "Asian"], default="Other or unknown")
    frame["ins_medicaid"] = (frame["insurance"].astype(str) == "Medicaid").astype(int)
    frame["lang_english"] = (frame["language"].astype(str) == "English").astype(int)
    frame["adm_urgent"] = (frame["admission_type"].astype(str) == "URGENT").astype(int)
    frame["loc_physician"] = (frame["admission_location"].astype(str) == "PHYSICIAN REFERRAL").astype(int)
    return frame

RED_BIN = ["ins_medicaid", "lang_english", "adm_urgent", "loc_physician"]
RED_CAT = ["race5"]
P_MAX_REDUCTION = 16

df = add_reduced_columns(df)
if "_make_spec_s15" not in globals():   # wrap only once, so re-running the cell does not recurse
    _make_spec_s15 = make_spec
RED15 = ["escalation_reduced", "escalation_landmark_reduced"]

def make_spec(name):
    if name not in RED15:
        return _make_spec_s15(name)
    sp = _make_spec_s15(name.replace("_reduced", ""))
    d = add_reduced_columns(sp["data"].copy())
    binc = [b for b in (["creatinine_missing"] + RED_BIN) if b in d.columns and d[b].nunique() > 1]
    return dict(name=name, data=d, num=list(sp["num"]), bin=binc, cat=list(RED_CAT), label=translate("y_esc", LANG))

P_RED = count_parameters(make_spec("escalation_reduced"))
print(f"Reduced set: {P_RED} parameters (maximum allowed {P_MAX_REDUCTION})")
assert P_RED <= P_MAX_REDUCTION, "reduced set exceeds the parameters supported by the data: STOP (do not modify after seeing results)"

RUN_SECTION_15 = True
if RUN_SECTION_15:
    for nm in RED15:
        results[nm] = run_analysis(nm, BOOT_SENS)
        emit(nm, lambda r=results[nm]: report(r))

def reduced_summary():
    print(f"SAMPLE-SIZE REDUCTION (G3) FOR ESCALATION (RETURN R2), QUICK={QUICK}, bootstrap B={BOOT_SENS}")
    print("Rule and reduced set: docs/decision_G3_reduction_2026-11-09.md (fixed before this run)")
    full, lm = results["escalation_reduced"], results["escalation_landmark_reduced"]
    n, ev = len(full["y"]), int(full["y"].sum())
    r2 = r2cs_from_oof(full)
    riley_block("escalation_reduced (label y_esc)", n, ev, P_RED, [("observed (cross-validated LR, reduced set)", r2)])
    p3_ok = riley(P_RED, ev / n, r2)["n_required"] <= n
    print(f"  -> gate G3 {'passes' if p3_ok else 'does NOT pass'}")
    rows = []
    for m in ("LR", "XGB", "Stack"):
        pf, pl = full["point"], lm["point"]
        lo = np.nanpercentile(full["boot"][(m, "auroc")], 2.5)
        prev_f, prev_l = full["y"].mean(), lm["y"].mean()
        gain_f, gain_l = pf[(m, "auprc")] / prev_f, pl[(m, "auprc")] / prev_l
        kept = (gain_l - 1) / (gain_f - 1) if gain_f > 1 else np.nan
        drop = pf[(m, "auroc")] - pl[(m, "auroc")]
        rows.append({"model": m, "events": ev,
                     "auroc": f"{pf[(m, 'auroc')]:.3f}" + _ci(full["boot"], (m, "auroc")),
                     "auprc": f"{pf[(m, 'auprc')]:.3f}" + _ci(full["boot"], (m, "auprc")),
                     "slope": f"{pf[(m, 'slope')]:.2f}", "O/E": f"{pf[(m, 'oe')]:.2f}",
                     "sens_top10": f"{pf[(m, 'sens_top10')]:.3f}",
                     "C2": "met" if (pf[(m, "auroc")] >= 0.75 and lo >= 0.70) else "not met",
                     "landmark auroc": f"{pl[(m, 'auroc')]:.3f}" + _ci(lm["boot"], (m, "auroc")),
                     "auroc drop": f"{drop:.3f}", "AUPRC gain kept": f"{kept:.2f}",
                     "C1": "met" if (drop <= 0.05 and kept >= 2 / 3) else "not met"})
    table(pd.DataFrame(rows))
    print("C3: not applicable. C4: notebook 09, section 10.")

if RUN_SECTION_15:
    emit("escalation_reduced_summary", reduced_summary)

## Exploratory (post hoc) — wider XGBoost grid for the reduced escalation model

Decided **after** seeing section 15 (XGBoost AUROC 0.749, just below C2). The XGBoost grid is widened to 36 configurations; everything else is unchanged and the configuration is still chosen only inside the inner folds. Reported as exploratory.

In [ ]:
GRID_XGB_SMALL = list(GRID["XGB"])
GRID_XGB_WIDE = [dict(depth=dp, spw=sp, lr=lr, mcw=mcw)
                 for dp in (1, 2, 3) for sp in (1, "sqrt") for lr in (0.02, 0.05, 0.1) for mcw in (5, 20)]
_make_clf_base = globals().get("_make_clf_base", make_clf)

def make_clf(name, cfg, ytr, seed):
    if name != "XGB" or "lr" not in cfg:
        return _make_clf_base(name, cfg, ytr, seed)
    pos = max(int(ytr.sum()), 1); neg = max(len(ytr) - pos, 1)
    spw = math.sqrt(neg / pos) if cfg["spw"] == "sqrt" else 1.0
    return XGBClassifier(n_estimators=XGB_TREES, learning_rate=cfg["lr"], max_depth=cfg["depth"], subsample=0.8,
                         colsample_bytree=0.8, min_child_weight=cfg["mcw"], scale_pos_weight=spw, eval_metric="logloss",
                         tree_method="hist", n_jobs=N_JOBS if N_JOBS and N_JOBS > 0 else 0, random_state=seed)

RUN_SECTION_16 = True
if RUN_SECTION_16:
    GRID["XGB"] = GRID_XGB_WIDE
    try:
        for nm in RED15:
            key = nm + "_xgbgrid"
            res = run_analysis(nm, BOOT_SENS)
            res["spec"] = dict(res["spec"], name=key)
            results[key] = res
            emit(key, lambda r=res: report(r))
    finally:
        GRID["XGB"] = GRID_XGB_SMALL   # restore the prespecified grid for any later section

def xgbgrid_summary():
    print(f"EXPLORATORY (post hoc): WIDER XGBOOST GRID, REDUCED ESCALATION MODEL, QUICK={QUICK}, bootstrap B={BOOT_SENS}")
    print(f"XGBoost grid: {len(GRID_XGB_WIDE)} configurations (prespecified grid: {len(GRID_XGB_SMALL)}); tuned inside inner folds only")
    full, lm = results["escalation_reduced_xgbgrid"], results["escalation_landmark_reduced_xgbgrid"]
    rows = []
    for m in ("LR", "XGB", "Stack"):
        pf, pl = full["point"], lm["point"]
        lo = np.nanpercentile(full["boot"][(m, "auroc")], 2.5)
        gain_f, gain_l = pf[(m, "auprc")] / full["y"].mean(), pl[(m, "auprc")] / lm["y"].mean()
        kept = (gain_l - 1) / (gain_f - 1) if gain_f > 1 else np.nan
        drop = pf[(m, "auroc")] - pl[(m, "auroc")]
        rows.append({"model": m, "auroc": f"{pf[(m, 'auroc')]:.3f}" + _ci(full["boot"], (m, "auroc")),
                     "auprc": f"{pf[(m, 'auprc')]:.3f}", "slope": f"{pf[(m, 'slope')]:.2f}", "O/E": f"{pf[(m, 'oe')]:.2f}",
                     "C2": "met" if (pf[(m, "auroc")] >= 0.75 and lo >= 0.70) else "not met",
                     "landmark auroc": f"{pl[(m, 'auroc')]:.3f}", "auroc drop": f"{drop:.3f}",
                     "gain kept": f"{kept:.2f}", "C1": "met" if (drop <= 0.05 and kept >= 2 / 3) else "not met"})
    table(pd.DataFrame(rows))
    cnt = pd.Series([c for n, c in full["chosen"] if n == "XGB"]).value_counts()
    print("\nXGBoost configurations chosen in the inner loops (full population):")
    table(cnt, index=True)

if RUN_SECTION_16:
    emit("escalation_reduced_xgbgrid_summary", xgbgrid_summary)

## Notebook summary

Criteria C1 to C3 with refitted models, and returns R4 and R2.

In [ ]:
# Summary of notebook 07
_rows = []
for _nm, _lab in [("landmark", "C1 landmark"), ("incident_documented", "C2 documented incident"),
                  ("incident_strict", "C2 strict incident"), ("aki_lab_incident", "C3 laboratory AKI, incident"),
                  ("labs_extended", "R4 extended laboratories"), ("escalation", "R2 escalation of care"),
                  ("escalation_reduced", "R2 escalation, reduced set (G3)")]:
    _rows.append((f"AUROC XGBoost, {_lab} (95% CI)", lambda nm=_nm: _m(results[nm], "XGB", "auroc")))
_rows.append(("classification at gate G4", "restated on the fitted models in notebook 10"))
notebook_summary(_rows)